# FLUX.1-schnell on free Colab T4 — 4-bit (memory-safe)

This is the **bulletproof** version for a free 16GB T4. It loads FLUX in **4-bit** (compressed), using ~8–10GB instead of ~24GB, so it won't run out of memory.

**Run order:** set `Runtime -> Change runtime type -> T4 GPU`, then run every cell top to bottom, once.

> If you ever hit an out-of-memory error, do **Runtime -> Restart session** and run again from the top. Never re-run after an error without restarting.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print('No GPU! Runtime -> Change runtime type -> T4 GPU')

## 2. Install dependencies
Includes `bitsandbytes` for 4-bit loading. ~2 min.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf bitsandbytes

## 3. Log in to Hugging Face
FLUX is gated. Create a free **Read** token at https://huggingface.co/settings/tokens and accept the license once at https://huggingface.co/black-forest-labs/FLUX.1-schnell — then paste the token below.

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

## 4. Load FLUX in 4-bit
Quantizes the two big pieces (the T5 text encoder and the transformer) to 4-bit so they fit the T4.
First run downloads the weights (a few minutes).

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBnb
from transformers import T5EncoderModel
from transformers import BitsAndBytesConfig as TransformersBnb

MODEL = "black-forest-labs/FLUX.1-schnell"

# 4-bit text encoder (T5-XXL — the biggest memory hog)
text_encoder_2 = T5EncoderModel.from_pretrained(
    MODEL, subfolder="text_encoder_2",
    quantization_config=TransformersBnb(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)

# 4-bit transformer (the image generator)
transformer = FluxTransformer2DModel.from_pretrained(
    MODEL, subfolder="transformer",
    quantization_config=DiffusersBnb(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)

pipe = FluxPipeline.from_pretrained(
    MODEL,
    text_encoder_2=text_encoder_2,
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
print("Pipeline ready.")

## 5. Generate an image
Edit the words in `prompt` and re-run this cell as many times as you like.

In [ ]:
prompt = "a cinematic wide shot of a lone cabin in the Tennessee smoky mountains at golden hour, mist in the valley, film grain"

image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    height=1024,
    width=1024,
    max_sequence_length=256,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save("flux_sample.png")
image

---
### If you still get out-of-memory
1. **Runtime -> Restart session**, then run top to bottom once.
2. In cell 5, lower `height` and `width` to `768` (or `512`).

### Notes
- 4-bit is slightly lower fidelity than full precision but runs on free hardware.
- For full quality + speed at volume, run the un-quantized model on a rented RTX 4090 (~$0.40/hr).